# **Norwegian News Articles**
Project for TDT4310

By: Malin Haugland Høli

## Topic Modeling using BERTopic

### *Imports*

In [ ]:
import pandas as pd
import numpy as np
import json

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired

c:\NTNU\V26\TDT4310\Project\exploring-norwegian-news-sources\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### *Import dataset*

In [2]:
df = pd.read_parquet("../combined-2019.parquet")
nob_df = df[df["language"] == "Bokmål"]

nob_df.head(3)

,file,url,source,date,author,gender,class1,class2,language,title,ingress,text,word_count,sentence_count
0,VG-20190101-16-personer-fikk-oye.xml,http://www.vg.no/nyheter/innenriks/i/jPyoyb/16...,VG,2019-01-01 09:12,Halvor Bjørntvedt,Male,"nyheter,innenriks",None,Bokmål,16 personer fikk øyeskader av fyrverkeri,Fem fikk alvorlige skader etter gårsdagens nyt...,"Tallene, som viser øyeskader på landsbasis, vi...",209,13
1,VG-20190101-47-plass-for-sundby-.xml,http://www.vg.no/sport/langrenn/i/3j8ra0/47-pl...,VG,2019-01-01 10:19,"Nils Mangelrød, Jostein Overvik",Male,"sport,langrenn",None,Bokmål,47. plass for Sundby - nekter å prate etter ne...,VAL MÜSTAIR (VG) Martin Johnsrud nektet å prat...,Den tidligere Tour de Ski-vinneren var svært l...,533,42
2,VG-20190101-94-drap-pa-journalis.xml,http://www.vg.no/nyheter/utenriks/i/J19kWP/94-...,VG,2019-01-01 21:37,Mikal Hem,Male,"nyheter,utenriks","Drap, Ytringsfrihet, Journalistikk, Korrupsjon...",Bokmål,94 drap på journalister i 2018: Skal ha blitt ...,Antall drepte journalister i Vesten har skutt ...,25. februar ble Jan Kuciak og hans forlovede M...,539,44


---
### *Defining stop words list*

The `stopwords-no.txt` file is downloaded from this Github repo: https://github.com/stopwords-iso/stopwords-no/blob/master/stopwords-no.txt. After several runs it became clear that adding those extra stopwords would be beneficial for the topic key words.

In [ ]:
# add custom stop words to CountVectorizer by reading from txt file
with open("../stopwords-no.txt", "r") as f:
    stop_words_no = f.read().splitlines()
    stop_words_no = [word.strip().lower() for word in stop_words_no if word.strip()]

extra_stopwords = [
    "flere", "siste", "tidligere", "blant", "annet",
    "sier", "opplyser", "skriver", "ifølge", "les", "litt"
]
extra_stopwords = extra_stopwords + [str(i) for i in range(1900, 2027)] # add years from 1900 to 2026 as stop words
extra_stopwords = extra_stopwords + [str(i) for i in range(1, 32)] # add numbers from 1 to 31 as stop words (for dates)
extra_stopwords = extra_stopwords + ["mandag", "tirsdag", "onsdag", "torsdag", "fredag", "lørdag", "søndag"] # add days of the week as stop words
extra_stopwords = extra_stopwords + [
    "aftenposten", "dagbladet", "nrk", "vg", "tv2", "bt", "dagsavisen", "adresseavisen", "adressa", 
    "fædrelandsvennen", "bergens tidende", "stavanger aftenblad", "dagens næringsliv", "dn", "nordlys"
    ] # add news sources as stop words
stop_words_no.extend(extra_stopwords)

print(f"Number of stop words: {len(stop_words_no)}")

Number of stop words: 221


---
### *Create Vectorizer*

In [ ]:
# Define vectorizer
ngram_range = (1, 3)
min_df = 10

# Create vectorizer with stopwords
vectorizer_model = CountVectorizer(
    stop_words=stop_words_no, 
    ngram_range=ngram_range,
    min_df=min_df
)
representation_model = KeyBERTInspired()

---
### *Load model*

In [5]:
print("Loading embedding model...")
model_name = "NbAiLab/nb-sbert-base"
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)
print("Embedding model loaded.")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1681.04it/s]
BertModel LOAD REPORT from: NbAiLab/nb-sbert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


### *Load parameters and fit model*

In [ ]:
sample_size = 50000

# Pass vectorizer into BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    verbose=True,
    min_topic_size=20
)

print("Sampling texts before fitting topic model...")
texts = nob_df.sample(sample_size, random_state=42)["text"].tolist()

print("Generating embeddings")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

print("Fitting topic model...")
topics, probs = topic_model.fit_transform(texts, embeddings)
print("Reducing number of topics.")
topic_model.reduce_topics(texts, nr_topics=20)
print("Topic model fitted.")

Fitting topic model...


2026-04-16 15:53:59,359 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-16 15:55:21,883 - BERTopic - Dimensionality - Completed ✓
2026-04-16 15:55:21,900 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-16 15:55:32,747 - BERTopic - Cluster - Completed ✓
2026-04-16 15:55:32,785 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-16 15:56:10,118 - BERTopic - Representation - Completed ✓


Reducing number of topics.


2026-04-16 15:56:39,252 - BERTopic - Topic reduction - Reducing number of topics
2026-04-16 15:56:39,680 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-16 15:57:14,775 - BERTopic - Representation - Completed ✓
2026-04-16 15:57:14,819 - BERTopic - Topic reduction - Reduced number of topics from 563 to 20


Topic model fitted.


### *Save model*

In [ ]:
ngram = ngram_range_str = f"{ngram_range[0]}-{ngram_range[1]}"
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

base_path = "../bertopic_models/NbAiLab_nb-sbert-base"
run_name = f"{model_name.replace('/', '_')}_ngram{ngram}_n{sample_size}_min_df{min_df}_{timestamp}"

save_path = f"{base_path}/{run_name}"

topic_model.save(save_path, serialization="safetensors")

config = {
    "model_name": model_name,
    "nr_topics": len(topic_model.get_topic_info()),
    "min_topic_size": topic_model.min_topic_size,
    "ngram_range": topic_model.n_gram_range,
    "min_df": "none",
    "sample_size": sample_size,
    "representation_model": "none",
    "vectorizer_model": "none"
}

# write to json file in the same directory as the model
with open(f"{save_path}/extended_config.json", "w") as f:
    json.dump(config, f, indent=4)

print(f"Model saved to {save_path}")

Model saved to ../bertopic_models/NbAiLab_nb-sbert-base/NbAiLab_nb-sbert-base_default_20260416_161124


---
### *Print topic info*

In [55]:
# Get topic info with counts and representative keywords
topic_info = topic_model.get_topic_info()

# Drop the outlier/noise topic (-1)
topic_info = topic_info[topic_info.Topic != -1]

# Sort topics by number of documents (Count) descending
top_topics = topic_info.sort_values(by="Count", ascending=False).head(10)

# Print top 10 topics with keywords
for _, row in top_topics.iterrows():
    topic_id = row["Topic"]
    keywords = topic_model.get_topic(topic_id)
    # Format keyword and score
    keywords_with_scores = [f"{kw} ({score:.3f})" for kw, score in keywords]
    print(f"Topic {topic_id} (Count: {row['Count']}): {keywords_with_scores}")

Topic 0 (Count: 125): ['prosent (0.021)', 'kroner (0.016)', 'på (0.015)', 'har (0.014)', 'av (0.014)', 'millioner (0.014)', 'og (0.013)', 'for (0.013)', 'er (0.013)', 'til (0.013)']
Topic 1 (Count: 105): ['norge (0.019)', 'vm (0.015)', 'det (0.014)', 'og (0.013)', 'er (0.013)', 'på (0.013)', 'de (0.013)', 'vi (0.013)', 'mot (0.012)', 'var (0.011)']
Topic 2 (Count: 105): ['united (0.029)', 'manchester (0.024)', 'solskjær (0.024)', 'liverpool (0.019)', 'league (0.015)', 'han (0.015)', 'var (0.015)', 'på (0.014)', 'etter (0.014)', 'mål (0.014)']
Topic 3 (Count: 103): ['trump (0.040)', 'usa (0.017)', 'han (0.015)', 'trumps (0.015)', 'at (0.015)', 'en (0.014)', 'som (0.013)', 'for (0.013)', 'til (0.013)', 'og (0.013)']
Topic 4 (Count: 78): ['iran (0.026)', 'syria (0.017)', 'is (0.016)', 'usa (0.016)', 'at (0.015)', 'og (0.015)', 'har (0.014)', 'som (0.014)', 'til (0.014)', 'de (0.014)']
Topic 5 (Count: 73): ['og (0.017)', 'av (0.016)', 'er (0.016)', 'for (0.016)', 'vi (0.015)', 'som (0.014)

In [58]:
topic_info = topic_model.get_topic_info()
num_topics = topic_info.Topic.nunique() - (1 if -1 in topic_info.Topic.values else 0)
noise_count = topic_info[topic_info.Topic == -1]["Count"].values[0] if -1 in topic_info.Topic.values else 0
avg_topic_size = topic_info[topic_info.Topic != -1]["Count"].mean()
print(f"Number of topics (excluding noise): {num_topics}")
print(f"Number of documents classified as noise: {noise_count}")
print(f"Average number of documents per topic (excluding noise): {avg_topic_size:.2f}")

# save number of topics, noise count, and average topic size to a text file
with open(f"{save_path}/topic_model_stats.txt", "w") as f:
    f.write(f"Number of documents in sample: {sample_size}\n")
    f.write(f"Number of documents classified as noise: {noise_count}\n")
    f.write(f"Percentage of documents classified as noise: {noise_count / sample_size:.2%}\n")
    f.write(f"Number of topics (excluding noise): {num_topics}\n")
    f.write(f"Average number of documents per topic (excluding noise): {avg_topic_size:.2f}\n")

Number of topics (excluding noise): 53
Number of documents classified as noise: 1076
Average number of documents per topic (excluding noise): 36.30


### *Visualize topic distribution*

In [ ]:
topic_model.visualize_topics()

In [ ]:
fig = topic_model.visualize_documents(texts, topics, embeddings)
fig.write_html(f"{save_path}/document_visualization.html")